# Piper Clean 100 试听测试

在 Colab 或本地 Jupyter 中从上到下运行。默认用 CPU 合成 **27 条 WAV**，用三个说话人 ID 和三档语速比较 `Hey Kamana` 的 B 版与 C 版，并保留 `hey hi` 试听。

`Hey Kamana` 的目标读音是 **hey kah-MAH-nah**。C 在 B 的 `MAH` 和弱读的 `nuh` 之间加了一个边界，试听时留意 `n` 是否更像最后一拍的开头。每组对照使用同一位说话人和同一档语速。

作者发布页写有 244 位说话人，随模型发布的配置列有 245 个 ID。这里试听 ID 0、122 和 244，验证末尾 ID 也能生成音频。音频保存到当前工作目录的 `piper_clean100_test/syllable_boundary_preview/`。

[Clean 100 权重与作者声明](https://brycebeattie.com/files/tts/) · [LibriTTS-R 数据来源](https://www.openslr.org/141/) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [1]:
%pip install "piper-tts==1.3.0"

In [2]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

from piper import PiperVoice

work_dir = Path.cwd() / "piper_clean100_test"
work_dir.mkdir(exist_ok=True)
source_url = "https://sfo3.digitaloceanspaces.com/bkmdls"
model_hashes = {
    "clean100.onnx": "a48b44084f5465a9d1c5e74c46c6ce6e1907af85e34f15292b9f44b13263f6e4",
    "clean100.onnx.json": "575a90e0f03154d30c4e8d9b1a86dcb590f7df2fc65e24466dde3c91491984b8",
}
for filename, expected_hash in model_hashes.items():
    path = work_dir / filename
    if path.is_file():
        if sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise ValueError(f"{path} 的 SHA-256 不匹配，请先移走该文件。")
    else:
        with TemporaryDirectory(dir=work_dir) as staging:
            downloaded = Path(staging) / filename
            urlretrieve(f"{source_url}/{filename}", downloaded)
            if sha256(downloaded.read_bytes()).hexdigest() != expected_hash:
                raise ValueError(f"{filename} 下载后的 SHA-256 不匹配。")
            downloaded.rename(path)

voice = PiperVoice.load(work_dir / "clean100.onnx")
print(f"ONNX 配置声明 {voice.config.num_speakers} 个说话人 ID，采样率 {voice.config.sample_rate} Hz。")

ONNX 配置声明 245 个说话人 ID，采样率 22050 Hz。


In [10]:
from statistics import median
from unicodedata import normalize
import wave

import numpy as np
from piper import SynthesisConfig

phrases = {
    "hey_kamana_b": "hˈeɪ kˌɑmˈanə",
    "hey_kamana_c": "hˈeɪ kˌɑmˈa nə",
    "hey_hi": "hˈeɪ hˈaɪ",
}
length_scales = {"fast": 0.65, "normal": 1.0, "slow": 1.25}
speaker_ids = (0, 122, 244)
wav_dir = work_dir / "syllable_boundary_preview"
wav_dir.mkdir(exist_ok=True)
if not set(speaker_ids) <= set(voice.config.speaker_id_map.values()):
    raise ValueError(f"存在模型配置未列出的说话人 ID。请检查 {speaker_ids}。")

wav_paths = {}
durations = {}
pcm_hashes = set()
for phrase, ipa in phrases.items():
    phonemes = list(normalize("NFD", ipa))
    missing = set(phonemes) - voice.config.phoneme_id_map.keys()
    if missing:
        raise ValueError(f"{phrase} 包含模型未定义的音素 {sorted(missing)}。")
    phoneme_ids = voice.phonemes_to_ids(phonemes)
    for speed, length_scale in length_scales.items():
        for speaker_id in speaker_ids:
            config = SynthesisConfig(
                speaker_id=speaker_id, length_scale=length_scale,
                noise_scale=0.667, noise_w_scale=0.8,
            )
            audio = voice.phoneme_ids_to_audio(phoneme_ids, config)
            peak = np.max(np.abs(audio), initial=0)
            if not np.isfinite(peak) or peak == 0:
                raise ValueError(f"{phrase} / {speed} / ID {speaker_id} 输出无效。")
            pcm = np.rint(audio / peak * 32767).astype("<i2").tobytes()
            path = wav_dir / f"{phrase}_{speed}_speaker_{speaker_id:03d}.wav"
            with wave.open(str(path), "wb") as wav:
                wav.setparams((1, 2, voice.config.sample_rate, 0, "NONE", ""))
                wav.writeframes(pcm)

            key = (phrase, speed, speaker_id)
            wav_paths[key] = path
            durations[key] = len(audio) / voice.config.sample_rate
            pcm_hashes.add(sha256(pcm).hexdigest())

expected = len(phrases) * len(length_scales) * len(speaker_ids)
assert len(wav_paths) == len(pcm_hashes) == expected, "试听样本缺失或 PCM 重复"
print(f"已生成 {expected} 条不同的试听 WAV，保存在 {wav_dir}。")
for phrase in phrases:
    for speed in length_scales:
        seconds = median(durations[phrase, speed, sid] for sid in speaker_ids)
        print(f"{phrase:10} {speed:6} 时长中位数 {seconds:.2f} 秒")

已生成 27 条不同的试听 WAV，保存在 /content/piper_clean100_test/syllable_boundary_preview。
hey_kamana_b fast   时长中位数 0.65 秒
hey_kamana_b normal 时长中位数 0.93 秒
hey_kamana_b slow   时长中位数 1.08 秒
hey_kamana_c fast   时长中位数 0.67 秒
hey_kamana_c normal 时长中位数 0.88 秒
hey_kamana_c slow   时长中位数 1.13 秒
hey_hi     fast   时长中位数 0.62 秒
hey_hi     normal 时长中位数 0.74 秒
hey_hi     slow   时长中位数 1.07 秒


In [11]:
from IPython.display import Audio, display

listen_speaker = 0  # 也可选 122 或 244
for speed, scale in length_scales.items():
    for phrase in phrases:
        key = (phrase, speed, listen_speaker)
        print(f"{phrase} · {speed} · length_scale={scale} · {durations[key]:.2f} 秒")
        display(Audio(filename=str(wav_paths[key]), normalize=False))

hey_kamana_b · fast · length_scale=0.65 · 0.65 秒


hey_kamana_c · fast · length_scale=0.65 · 0.65 秒


hey_hi · fast · length_scale=0.65 · 0.53 秒


hey_kamana_b · normal · length_scale=1.0 · 0.93 秒


hey_kamana_c · normal · length_scale=1.0 · 0.85 秒


hey_hi · normal · length_scale=1.0 · 0.72 秒


hey_kamana_b · slow · length_scale=1.25 · 1.08 秒


hey_kamana_c · slow · length_scale=1.25 · 1.13 秒


hey_hi · slow · length_scale=1.25 · 1.17 秒
